# FraudShield - Week 1: Exploratory Data Analysis (EDA)

Welcome to the **FraudShield** exploratory data analysis workspace.

### Objectives of this Notebook:
1. Inspect the dataset structure and verify the schema.
2. Quantify the severe class imbalance ($0.172\%$ positive fraud rate).
3. Analyze transaction amounts and temporal distribution.
4. Discover correlations between PCA features ($V1 - V28$) and fraudulent behavior.
5. Formulate modeling strategies for Week 2 (Baseline Logistic Regression).

In [ ]:
import sys
from pathlib import Path

# Add repository root to python search path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import load_config
from src.data.loader import load_raw_data, get_dataset_summary

sns.set_theme(style="whitegrid", palette="muted")
print("Environment initialized successfully!")

## 1. Load Data and Check Summary

In [ ]:
config = load_config()
raw_data_path = project_root / config["data"]["raw_path"]
print(f"Target dataset path: {raw_data_path}")

df = load_raw_data(raw_data_path)
summary = get_dataset_summary(df)
print("Dataset Summary:", summary)

## 2. Class Imbalance Analysis
In fraud detection, **Accuracy** is completely misleading.
A model predicting '0' (genuine) for all transactions would achieve **99.83% accuracy**, yet catch **0% of fraud**.

In [ ]:
class_counts = df['Class'].value_counts()
print(class_counts)

plt.figure(figsize=(6, 4))
sns.countplot(x='Class', data=df, hue='Class', palette=['#2ecc71', '#e74c3c'], legend=False)
plt.yscale('log')
plt.title('Class Counts (Log Scale)')
plt.show()

## 3. Transaction Amount Analysis
Comparing spending patterns between legitimate cardholders and fraudulent actors.

In [ ]:
plt.figure(figsize=(10, 4))
sns.kdeplot(df[df['Class'] == 0]['Amount'], label='Genuine', fill=True, color='#2ecc71', clip=(0, 500))
sns.kdeplot(df[df['Class'] == 1]['Amount'], label='Fraud', fill=True, color='#e74c3c', clip=(0, 500))
plt.title('Kernel Density of Transaction Amounts ($0 to $500)')
plt.xlabel('Amount ($)')
plt.legend()
plt.show()

## 4. PCA Features Correlation with Fraud
Features V1 to V28 were derived using PCA from the raw transaction attributes for confidentiality.
Which components have the strongest linear correlation with the `Class` target?

In [ ]:
correlations = df.corr()['Class'].drop('Class').sort_values()
top_corrs = pd.concat([correlations.head(7), correlations.tail(7)])

plt.figure(figsize=(10, 5))
top_corrs.plot(kind='bar', color=['#e74c3c' if val > 0 else '#3498db' for val in top_corrs])
plt.title('Top Positively and Negatively Correlated Features with Fraud')
plt.ylabel('Pearson Correlation')
plt.show()